# Computer Vision Notebook 2: Image Sampling, Aliasing, and Mosaic Images

**Goal:** understand why image size, sampling rate, and anti-aliasing matter.

Aliasing happens when we sample a pattern too sparsely. The computer records a false-looking pattern that was not really in the original signal. This is important in cameras, screens, satellite images, microscopes, greenhouse sensors, and any computer vision model.

In [ ]:
!pip -q install numpy matplotlib scipy pillow scikit-image

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter
from PIL import Image, ImageDraw
from skimage.transform import resize

## 1. Aliasing in a 1D signal

A camera samples the world at discrete pixel locations. This first example uses a wave instead of an image so the idea is easier to see.

In [ ]:
x = np.linspace(0, 1, 1000)
frequency = 18
signal = np.sin(2*np.pi*frequency*x)

xs_good = np.linspace(0, 1, 80)
ys_good = np.sin(2*np.pi*frequency*xs_good)
xs_bad = np.linspace(0, 1, 25)
ys_bad = np.sin(2*np.pi*frequency*xs_bad)

plt.figure(figsize=(10, 4))
plt.plot(x, signal, label='original signal')
plt.scatter(xs_good, ys_good, s=18, label='enough samples')
plt.scatter(xs_bad, ys_bad, s=40, marker='x', label='too few samples')
plt.title('Sampling a high-frequency pattern')
plt.xlabel('position')
plt.ylabel('brightness')
plt.legend();

## 2. Aliasing in images

A checkerboard has many rapid changes. When it is downsampled without smoothing, it can create strange patterns.

In [ ]:
n = 256
xx, yy = np.meshgrid(np.arange(n), np.arange(n))
checker = (((xx // 4) + (yy // 4)) % 2).astype(float)

small_no_antialias = checker[::8, ::8]
blurred = gaussian_filter(checker, sigma=3.0)
small_with_antialias = blurred[::8, ::8]

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
images = [checker, small_no_antialias, small_with_antialias]
titles = ['Original high-frequency pattern', 'Downsampled directly', 'Blur before downsampling']
for ax, im, title in zip(axes, images, titles):
    ax.imshow(im, cmap='gray', interpolation='nearest')
    ax.set_title(title)
    ax.axis('off')
plt.tight_layout()

## 3. Image resizing with anti-aliasing

Many libraries can blur automatically before downsampling. Try changing `anti_aliasing=True` to `False`.

In [ ]:
small_skimage = resize(checker, (32, 32), anti_aliasing=True)
small_no = resize(checker, (32, 32), anti_aliasing=False, order=0)

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
for ax, im, title in zip(axes, [small_no, small_skimage], ['No anti-aliasing', 'With anti-aliasing']):
    ax.imshow(im, cmap='gray', interpolation='nearest')
    ax.set_title(title)
    ax.axis('off')
plt.tight_layout()

## 4. Original image vs mosaic image

A mosaic image is made by reducing the resolution, then scaling back up with nearest-neighbor interpolation. It is useful for privacy, art, and teaching pixel structure.

In [ ]:
# Make a simple synthetic scene so the notebook works without external files.
W, H = 420, 280
img = Image.new('RGB', (W, H), (238, 246, 240))
d = ImageDraw.Draw(img)
for y in range(H):
    val = int(220 - 70*y/H)
    d.line([(0, y), (W, y)], fill=(val, min(val+20, 235), 245))
for i in range(6):
    x0 = 30 + i*62
    d.rectangle([x0, 170, x0+50, 220], fill=(92,144,91), outline=(45,85,55), width=2)
    for j in range(8):
        cx = x0 + 8 + (j % 4)*11
        cy = 180 + (j // 4)*16
        d.ellipse([cx-5, cy-5, cx+5, cy+5], fill=(36,125,62))
for p in [(90,185), (210,198), (330,178), (366,203)]:
    d.ellipse([p[0]-4, p[1]-4, p[0]+4, p[1]+4], fill=(200,60,45))

small = img.resize((42, 28), Image.Resampling.BILINEAR)
mosaic = small.resize((W, H), Image.Resampling.NEAREST)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, im, title in zip(axes, [img, mosaic], ['Original', 'Mosaic / pixelated']):
    ax.imshow(im)
    ax.set_title(title)
    ax.axis('off')
plt.tight_layout()

## 5. Challenge

1. Change the mosaic size from `(42, 28)` to `(84, 56)` and `(21, 14)`.
2. What happens when the mosaic blocks become larger?
3. Why might face datasets or street-view datasets use mosaic or blur for privacy?